In [4]:
import sys
sys.path.append('/data/user/wu_j1/bamf')

In [11]:
import numpy as np
import pandas as pd

from analysis_codes.itx_helper import *

folder = "C:/Users\wu_j1-adm\switchdrive/"

folder = "D:/work/Data/"
folder = "/data/user/wu_j1/"

filename = "Zurich-ToF-ACSM/raw/input_matrices_martavia_noised_20220315.itx"
filename = "Krakow/raw/KRK_PMF_input_new.itx"
# filename="Krakow/raw/KRK_PMF_ts.itx"
mz2remove = [16,17,18,19,20,28,12,37]

# read matrix itx file

In [6]:
waves = itx_to_pandas(folder + filename)
waves.keys()


dict_keys(['WAVES/D'])

In [7]:
waves['WAVES/D']
df = waves['WAVES/D']
df

,DateUTC,HOA,BBOA,CCOA,MO_OOA,LO_OOA,OOA
0,3.598270e+09,1.088912,0.502180,1.128500,1.522545,0.383934,1.906480
1,3.598272e+09,1.037656,0.020797,1.303923,0.680600,0.384180,1.064781
2,3.598274e+09,0.849802,0.670091,1.091423,0.938110,0.836266,1.774376
3,3.598276e+09,1.065534,0.239267,0.944973,1.499861,0.695237,2.195098
4,3.598277e+09,1.006902,0.188146,1.134418,2.003491,0.947114,2.950605
...,...,...,...,...,...,...,...
16268,3.637717e+09,0.878920,0.825180,0.727054,5.379306,4.467822,9.847128
16269,3.637719e+09,0.744441,0.425404,0.735912,5.869260,4.158776,10.028036
16270,3.637721e+09,0.978829,0.567442,0.767273,5.243870,3.730336,8.974206
16271,3.637723e+09,0.830700,0.394445,0.807225,5.543607,4.784796,10.328404


In [8]:
df["DateUTC"] = df["DateUTC"].apply(igor_time_convert)
# igor_time_convert

In [6]:
df

,DateUTC,HOA,BBOA,CCOA,MO_OOA,LO_OOA,OOA
0,2018-01-08 15:30:00,1.088912,0.502180,1.128500,1.522545,0.383934,1.906480
1,2018-01-08 16:00:00,1.037656,0.020797,1.303923,0.680600,0.384180,1.064781
2,2018-01-08 16:30:00,0.849802,0.670091,1.091423,0.938110,0.836266,1.774376
3,2018-01-08 17:00:00,1.065534,0.239267,0.944973,1.499861,0.695237,2.195098
4,2018-01-08 17:30:00,1.006902,0.188146,1.134418,2.003491,0.947114,2.950605
...,...,...,...,...,...,...,...
16268,2019-04-10 05:00:00,0.878920,0.825180,0.727054,5.379306,4.467822,9.847128
16269,2019-04-10 05:30:00,0.744441,0.425404,0.735912,5.869260,4.158776,10.028036
16270,2019-04-10 06:00:00,0.978829,0.567442,0.767273,5.243870,3.730336,8.974206
16271,2019-04-10 06:30:00,0.830700,0.394445,0.807225,5.543607,4.784796,10.328404


# read vector itx file

In [13]:
waves = itx_to_pandas(folder + filename)
waves.keys()

dict_keys(['amus', 'Org_Specs_30', 'OrgSpecs_err_30', 'acsm_utc_time_30'])

In [14]:
Specs = waves['Org_Specs_30']
Specs.shape

(16286, 93)

In [15]:
error = waves['OrgSpecs_err_30']
error.shape

(16286, 93)

In [16]:
amus_nb = waves['amus']
amus_nb.shape

(93, 1)

In [17]:
timelist = waves['acsm_utc_time_30']
timelist.shape

(16286, 1)

In [18]:
Specs = remove_mz(Specs,amus_nb,mz2remove)
error = remove_mz(error,amus_nb,mz2remove)

In [21]:
time = igor_time_convert(timelist)
time

array(['2018-01-08T15:30:00', '2018-01-08T16:00:00',
       '2018-01-08T16:30:00', ..., '2019-04-10T06:00:00',
       '2019-04-10T06:30:00', '2019-04-10T07:00:00'],
      dtype='datetime64[s]')

In [23]:
amus_nb = np.array(([int(i) for i in amus_nb[0] if i not in mz2remove]))

In [24]:
amus_nb.shape,Specs.shape

((88,), (16286, 88))

In [25]:

Specs.columns=[int(i) for i in amus_nb]
Specs.index = time

# Specs.to_hdf(folder+'/Krakow/processed/KRK_PMF_input_new.h5', key='data', mode='w')  

In [26]:

error.columns=[int(i) for i in amus_nb]
error.index = time
# error
# error.to_hdf(folder+'/Krakow/processed/KRK_PMF_input_new.h5', key='error', mode='r+')  

In [ ]:
error

In [ ]:
# filepath = "work\Data/Zurich-ToF-ACSM\input_matrices_martavia_noised_20220315.h5"
# error_df = pd.read_hdf(folder+filepath,"error")
# data_df = pd.read_hdf(folder+filepath,"data")

In [ ]:
winter_data_df = Specs[(Specs.index <= pd.Timestamp(2018, 10, 24, 0)) &(Specs.index >= pd.Timestamp(2018, 10, 10, 0))]
winter_error_df = error[(error.index <= pd.Timestamp(2018, 10, 24, 0)) &(error.index >= pd.Timestamp(2018, 10, 10, 0))]

winter_data_df = Specs[(Specs.index <= pd.Timestamp(2019, 4, 11 , 0)) &(Specs.index >= pd.Timestamp(2019, 3, 15,0))]
winter_error_df = error[(error.index <= pd.Timestamp(2019, 4, 11 , 0)) &(error.index >= pd.Timestamp(2019, 3, 15,0))]

# winter_data_df.to_hdf(folder+'/Krakow/processed/KRK_PMF_input_Mar.h5', key='data', mode='w')  
# winter_error_df.to_hdf(folder+'/Krakow/processed/KRK_PMF_input_Mar.h5', key='error', mode='r+')  

In [ ]:
winter_data_df

In [ ]:
winter_data_df.to_numpy().min(),winter_data_df.to_numpy().max()

# read generated h5 data file

In [ ]:
import h5py
f = h5py.File( "/data/user/wu_j1/Krakow/processed/KRK_PMF_input_Oct.h5",'r')

keys = list(f.keys())
keys

In [ ]:
data_df = pd.read_hdf("/data/user/wu_j1/Krakow/processed/KRK_PMF_input_Oct.h5",key='data')
data_df